# kAIsparov — Run analysis & interpretability

A scratchpad for the data-science side of the project: read the training runs
recorded under `runs/`, plot learning curves, and — the deep goal — **probe what
the GNN has actually learned** (edge/move scores, node embeddings, attention).

Requires the package installed (`pip install -e .`) and the notebook extras
(`pip install -e '.[notebooks]'`).


## Setup


In [2]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

from kaisparov.tracking.registry import Registry

# Locate the repo's runs/ dir whether the notebook runs from repo root or notebooks/.
ROOT = Path.cwd()
RUNS_DIR = ROOT / 'runs' if (ROOT / 'runs').exists() else ROOT.parent / 'runs'
registry = Registry(RUNS_DIR)
runs = registry.list_runs()
print(f'{len(runs)} run(s) under {RUNS_DIR}')
pd.DataFrame(
    [{'run_id': r['run_id'], 'model': r['model'], 'status': r['status'],
      'epochs': r['epochs_completed'], 'params': r.get('num_params')} for r in runs]
)

0 run(s) under c:\Users\lucas\OneDrive\Projects\kAIsparov\runs


""


## Learning curves

Pick a run and plot its per-epoch training metrics.


In [ ]:
def load_metrics(run_id: str) -> pd.DataFrame:
    path = RUNS_DIR / run_id / 'metrics.jsonl'
    rows = [pd.read_json(line, typ='series') for line in path.read_text().splitlines()]
    return pd.DataFrame(rows)

run_id = runs[0]['run_id']  # newest run; change as needed
df = load_metrics(run_id)
train = df[df['section'] == 'train']

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].plot(train['epoch'], train['loss'], label='loss')
ax[0].plot(train['epoch'], train['policy_loss'], label='policy')
ax[0].plot(train['epoch'], train['value_loss'], label='value')
ax[0].set_xlabel('epoch'); ax[0].set_title('losses'); ax[0].legend()
ax[1].plot(train['epoch'], train['entropy'], color='tab:purple')
ax[1].set_xlabel('epoch'); ax[1].set_title('policy entropy')
fig.suptitle(run_id); fig.tight_layout()

## Elo vs baselines (across a resume lineage)

Follow the run's lineage so a resumed run continues the same curve.


In [ ]:
lineage = registry.lineage(run_id)
evals = pd.concat([pd.DataFrame(r.get('eval_history', [])) for r in lineage], ignore_index=True)

if not evals.empty:
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.plot(evals['epoch'], evals['elo_vs_random'], marker='o', label='vs random')
    ax.plot(evals['epoch'], evals['elo_vs_material'], marker='o', label='vs material')
    ax.axhline(0, color='grey', lw=0.7)
    ax.set_xlabel('epoch'); ax.set_ylabel('Elo diff'); ax.set_title('Elo vs baselines')
    ax.legend()
else:
    print('No eval history yet — train with eval enabled.')

## Interpretability (the main goal)

Load a checkpoint and inspect what the policy does on a concrete position: which
moves (edges) it scores highest, and the critic's value. This cell is the seed for
deeper probes — node embeddings, per-relation message flow, attention weights.


In [ ]:
import torch

from kaisparov.core.board import ChessGame
from kaisparov.core.utils import index_to_coord
from kaisparov.models.factory import load_backend_spec

spec = load_backend_spec('gnn_v1')
device = torch.device('cpu')
ckpt = registry.resolve_checkpoint(run_id)  # latest checkpoint by default
model, _ = spec.model_class.load_agent_for_inference(device=device, model_path=ckpt, hidden_dim=8)
processor = spec.processor_class()

game = ChessGame()  # standard start; try a curriculum position too
data = processor.graphify(game)
with torch.no_grad():
    action_scores, value = model(data)

edge_index = data.edge_index
topk = torch.topk(action_scores, k=8).indices
print(f'state value = {float(value):.4f}')
print('top-scored moves (may be illegal — no mask applied here):')
for e in topk:
    s = index_to_coord(int(edge_index[0, e])); d = index_to_coord(int(edge_index[1, e]))
    print(f'  {s} -> {d}   score={float(action_scores[e]):+.3f}')